In [ ]:
'''
cd .venv/lib/python3.13/site-packages/pyspark/sbin

SPARK_LOCAL_IP=127.0.0.1 ./spark-daemon.sh start org.apache.spark.deploy.master.Master 1
starting org.apache.spark.deploy.master.Master, logging to /home/maxim_savrilov/spark/.venv/lib/python3.13/site-packages/pyspark/logs/spark-maxim_savrilov-org.apache.spark.deploy.master.Master-1-instance-20260811-092123.out
SPARK_LOCAL_IP=127.0.0.1 ./spark-daemon.sh start org.apache.spark.deploy.worker.Worker 1 spark://127.0.0.1:7077
starting org.apache.spark.deploy.worker.Worker, logging to /home/maxim_savrilov/spark/.venv/lib/python3.13/site-packages/pyspark/logs/spark-maxim_savrilov-org.apache.spark.deploy.worker.Worker-1-instance-20260811-092123.out

'''

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [2]:
# 1. Привязываем правильный, стабильный shaded-джарник
conf = SparkConf() \
    .setMaster('spark://127.0.0.1:7077') \
    .setAppName('test') \
    .set("spark.jars", "./lib/gcs-connector-4.0.4-shaded.jar")

sc = SparkContext(conf=conf)

# 2. Прописываем оригинальные классы Google для работы с протоколом gs://
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")

hadoop_conf.set("fs.gs.block.size", "67108864")

26/08/13 12:07:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
# 3. Стартуем стандартную сессию Spark SQL (без отключения векторизации!)
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [5]:
spark

In [6]:
df_green = spark.read.option("recursiveFileLookup", "true").parquet('gs://maksim-savrilov-spark/pq/green/')

In [7]:
df_green.limit(5).toPandas()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2.0,2020-01-23 13:10:15,2020-01-23 13:38:16,N,1.0,74,130,1.0,12.77,36.0,0.00,0.5,2.05,6.12,NaN,0.3,44.97,1.0,1.0,0.0
1,NaN,2020-01-20 15:09:00,2020-01-20 15:46:00,None,NaN,67,39,NaN,8.00,29.9,2.75,0.5,0.00,0.00,NaN,0.3,33.45,NaN,NaN,NaN
2,2.0,2020-01-15 20:23:41,2020-01-15 20:31:18,N,1.0,260,157,1.0,1.27,7.0,0.50,0.5,0.00,0.00,NaN,0.3,8.30,2.0,1.0,0.0
3,2.0,2020-01-05 16:32:26,2020-01-05 16:40:51,N,1.0,82,83,1.0,1.25,7.5,0.00,0.5,0.00,0.00,NaN,0.3,8.30,2.0,1.0,0.0
4,2.0,2020-01-29 19:22:42,2020-01-29 19:31:02,N,1.0,166,42,1.0,1.84,8.0,1.00,0.5,2.94,0.00,NaN,0.3,12.74,1.0,1.0,0.0
